# Fase 6-7: Evaluasi dan Interpretasi Model (XAI)
Berdasarkan hasil Fase 3.5 & 5, **XGBoost Regressor** ditetapkan sebagai pemenang untuk dievaluasi secara final (termasuk Bootstrapping 95% CI dan interpretasi lewat **SHAP value**).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib

from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## 1. Load Dataset dan Split Test Set

In [ ]:
df = pd.read_csv('../data/processed/model_ready_data.csv')
X = df.drop(columns=['log_maxpga'])
y = df['log_maxpga']

# Melakukan Split Test Set (HANYA SEKALI DIGUNAKAN DI FASE INI)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Ukuran Train:", X_train.shape)
print("Ukuran Test:", X_test.shape)

## 2. Latih Ulang Model Pemenang (XGBoost)

In [ ]:
# Parameter diambil dari performa yang dievaluasi di literatur SOTA / Fase Modeling
best_model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1)
best_model.fit(X_train, y_train)

# Prediksi di Test Set
y_pred = best_model.predict(X_test)

# Metrik Utama
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE : {mae:.4f}")
print(f"Test R^2 : {r2:.4f}")

## 3. Evaluasi Statistika Lanjut: Bootstrapping 95% Confidence Interval
Untuk memastikan bahwa performa metrik yang dimenangkan bukan karena beruntung (lucky random split).

In [ ]:
n_iterations = 1000
bootstrapped_r2 = []

test_indices = np.arange(len(y_test))
y_test_arr = y_test.values
y_pred_arr = y_pred

np.random.seed(42)
for i in range(n_iterations):
    boot_indices = np.random.choice(test_indices, size=len(test_indices), replace=True)
    y_test_boot = y_test_arr[boot_indices]
    y_pred_boot = y_pred_arr[boot_indices]
    
    score = r2_score(y_test_boot, y_pred_boot)
    bootstrapped_r2.append(score)

# Menghitung Batas
percentile_2_5 = np.percentile(bootstrapped_r2, 2.5)
percentile_97_5 = np.percentile(bootstrapped_r2, 97.5)

print(f"95% Confidence Interval for R^2: [{percentile_2_5:.4f}, {percentile_97_5:.4f}]")

plt.figure(figsize=(8, 5))
sns.histplot(bootstrapped_r2, bins=30, kde=True)
plt.axvline(percentile_2_5, color='red', linestyle='dashed', label='2.5th Pctl')
plt.axvline(percentile_97_5, color='red', linestyle='dashed', label='97.5th Pctl')
plt.title("Bootstrapped R² Distribution (1000 iter)")
plt.legend()
plt.show()

## 4. Analisis Explainable AI (XAI) Menggunakan SHAP

In [ ]:
# Inisialisasi explainer untuk Tree Model (XGBoost)
explainer = shap.TreeExplainer(best_model)
shap_values = explainer(X_test)

# 1. Summary Plot (Pengaruh fitur secara Global)
plt.title("SHAP Global Summary")
shap.summary_plot(shap_values, X_test)

In [ ]:
# 2. Waterfall Plot (Pengaruh fitur secara Lokal pada pengamatan stasiun tertentu)
# Kita cari observasi dengan PGA (Ground Motion) terbesar di test set
idx_max_pga = np.argmax(y_test.values)

plt.title(f"SHAP Waterfall (Prediksi untuk Test Record Index: {idx_max_pga})")
shap.waterfall_plot(shap_values[idx_max_pga], max_display=10)

## 5. Ekspor Model Q1 
Menyimpan model yang divalidasi penuh ke bentuk serial objek, agar siap dipanggil (deployment).

In [ ]:
model_export_path = '../data/processed/pga_xgboost_model.joblib'
joblib.dump(best_model, model_export_path)
print(f"Model XGBoost sukses diekspor ke: {model_export_path}")